In [1]:
import os

os.environ["NET"] = "TUNNEL"

In [2]:
from elasticsearch_dsl import Search
from loguru import logger
from pymongo import UpdateOne

from dm.connector.elasticsearch.manager import get_es_with_alias_name
from dm.connector.mongo.manager2 import get_database_with_alias_name

ModuleNotFoundError: No module named 'elasticsearch_dsl'

In [ ]:
site = "ml_br"

es, index_name = get_es_with_alias_name(f"main_{site}")

question_ml_mx_db = get_database_with_alias_name("question_ml_mx")
question_db = question_ml_mx_db.client.get_database(name=site)
ci_sku_pools_table = question_db.get_collection(name="ci_sku_pools")
ci_sku_pools_table.create_index("sl")

{'name': 'VM-1-12-ubuntu', 'cluster_name': 'elasticsearch', 'cluster_uuid': '1rA73wYhRI29cr4nWxtMbA', 'version': {'number': '7.10.0', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '51e9d6f22758d0374a0f3f5c6e8f3a7997850f96', 'build_date': '2020-11-09T21:30:33.964949Z', 'build_snapshot': False, 'lucene_version': '8.7.0', 'minimum_wire_compatibility_version': '6.8.0', 'minimum_index_compatibility_version': '6.0.0-beta1'}, 'tagline': 'You Know, for Search'}
['sku_pools_ys', 'visit_daily_task_current_od30', 'sku_oppo_20240925_tmp']


'sl_1'

In [ ]:
es.get(id="MLM1509240886", index=index_name)

{'_index': 'mlmx_sku',
 '_type': '_doc',
 '_id': 'MLM1509240886',
 '_version': 585,
 '_seq_no': 6768510417,
 '_primary_term': 40,
 'found': True,
 '_source': {'st': 50,
  'spu_url': 'https://articulo.mercadolibre.com.mx/MLM-1509240886-maquina-antironquidos-para-apnea-del-sueno-ayuda-a-dormir-_JM',
  'pr': 525.46,
  'rv': 19,
  'os': 0,
  'sy': 'ci',
  'sellerName': 'NATURALMALLMX2',
  'best': 0,
  'title': 'Máquina Antironquidos Para Apnea Del Sueño, Ayuda A Dormir,',
  'sid': 'MLM1509240886',
  'puton': 20220922,
  'total_order': 0,
  'br': 'Genérica',
  'catid': 'MLM194398',
  'datetime': 1735194342515,
  'sellerID': '622470003',
  'cbt': 2,
  'rate': 2.4,
  'cat': {'cat_1': 'Deportes y Fitness ',
   'cat_2': 'Natación',
   'cat_3': 'Accesorios para Natación',
   'cat_4': 'Clips de Naríz'},
  'rank': 296,
  'pic_url': 'https://http2.mlstatic.com/D_NQ_NP_918069-CBT51641798930_092022-O.webp',
  'on': 0,
  'od30': 0,
  'od7': 0,
  'od14': 0,
  'monthly_sale_trend': {'2209': 0},
  'profi

In [ ]:
for i in es.cat.indices(format="json"):
    print(i)

{'health': 'yellow', 'status': 'open', 'index': 'mlmx_sku_2', 'uuid': 'u0ikrt-4SQuUeUdoH-VfvQ', 'pri': '1', 'rep': '1', 'docs.count': '0', 'docs.deleted': '0', 'store.size': '208b', 'pri.store.size': '208b'}
{'health': 'green', 'status': 'open', 'index': '.apm-custom-link', 'uuid': '0gLDoXCHRWu9bXk25TaHLA', 'pri': '1', 'rep': '0', 'docs.count': '0', 'docs.deleted': '0', 'store.size': '208b', 'pri.store.size': '208b'}
{'health': 'green', 'status': 'open', 'index': '.kibana_task_manager_1', 'uuid': 'ysoKmbYqQwuCSeGkkYkvyA', 'pri': '1', 'rep': '0', 'docs.count': '5', 'docs.deleted': '57727', 'store.size': '4.6mb', 'pri.store.size': '4.6mb'}
{'health': 'green', 'status': 'open', 'index': '.monitoring-es-7-2025.03.23', 'uuid': '99Rpv_hZQjuiLYMdRGn3og', 'pri': '1', 'rep': '0', 'docs.count': '259838', 'docs.deleted': '18048', 'store.size': '129.2mb', 'pri.store.size': '129.2mb'}
{'health': 'green', 'status': 'open', 'index': '.monitoring-es-7-2025.03.22', 'uuid': 'RIRQsXfdQvKV192TzDLwuA', 'pr

In [ ]:
s = Search(using=es, index=index_name)
s: Search = s.filter("term", **{"sy.keyword": "ci"})
s: Search = s.filter("term", os=0)
s: Search = s.filter("term", on=0)
s: Search = s.filter("range", ot={"gte": 5})
s.count()

214464

In [ ]:
s = Search(using=es, index=index_name).params(scroll="10m", request_timeout=60)
s: Search = s.filter("term", sy="ci")
s: Search = s.filter("term", os="0")
s: Search = s.filter("term", on=0)
s: Search = s.filter("range", ot={"gte": 25})
s: Search = s.source(include=["sid", "ot"])
# s: Search = s.sort("-ot")
response = s.scan()

n = 0
requests_update = []
for hit in response:
    n += 1
    _data = hit.to_dict()
    if "#" in _data["sid"]:
        continue
    # print(_data)
    sku_id = _data["sid"]
    requests_update.append(UpdateOne(filter={"sl": sku_id}, update={"$set": {"dt": 20250312, "rst": 0}}, upsert=True))

    if n % 1000 == 0:
        logger.info(f"{n=}")
        ci_sku_pools_table.bulk_write(requests=requests_update)
        requests_update.clear()

    # if n >= 50_0:
    #     break

if requests_update:
    logger.info(f"{n=}")
    ci_sku_pools_table.bulk_write(requests=requests_update)
    requests_update.clear()

2025-03-12 10:59:11.659 | INFO     | __main__:<module>:22 - n=1000
2025-03-12 10:59:15.873 | INFO     | __main__:<module>:22 - n=2000
2025-03-12 10:59:17.525 | INFO     | __main__:<module>:30 - n=2271


In [ ]:
from importlib.metadata import version

numpy_version = version("dm")
print(numpy_version)


2025.3.21.1
